##Part 1 — Python & Data Foundations: ingest, validate, clean, enrich


In [11]:
import csv, json
import time, requests
import pandas as pd
import numpy as np


Task 1.1 — Read the file the foundations way

In [12]:
def parse_row(row):
    """Converts specific numerical fields to int and float."""
    parsed = row.copy()
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

accepted = []
rejected = []
filename = "data/appointments.csv"

with open(filename, mode="r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    print(f"Header: {header}")

    row_count = 0
    for row in reader:
        row_count += 1
        try:
            converted_row = parse_row(row)
            accepted.append(converted_row)
        except ValueError:
            rejected.append(row)

    print(f"Total rows processed: {row_count}")

with open("rejected_rows.json", mode="w", encoding="utf-8") as jf:
    json.dump(rejected, jf, indent=2)

print(f"Accepted rows: {len(accepted)}")
print(f"Rejected rows: {len(rejected)} (saved to rejected_rows.json)")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows processed: 1020
Accepted rows: 988
Rejected rows: 32 (saved to rejected_rows.json)


Task 1.2 — Inspect and clean, pass 1

In [13]:
df = pd.read_csv("data/appointments.csv")

print(f"Shape: {df.shape} | Duplicates: {df.duplicated().sum()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print("\nDataset Info:")
df.info()

# Step (a): Drop exact duplicate rows
df = df.drop_duplicates()
print(f"\nRow count after dropping exact duplicates: {len(df)}")

# Step (b): Clean booking_channel with .str.strip().str.title()
if "booking_channel" in df.columns:
    df["booking_channel"] = df["booking_channel"].astype(str).str.strip().str.title()
print(f"Row count after cleaning booking_channel: {len(df)}")

# Step (c): Convert appt_date and booked_on to datetime
if "appt_date" in df.columns:
    df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
if "booked_on" in df.columns:
    df["booked_on"] = pd.to_datetime(df["booked_on"], errors="coerce")
print(f"Row count after datetime conversion: {len(df)}")

# Step (d): Drop rows where appt_date is missing or status is missing
subset_cols = [col for col in ["appt_date", "status"] if col in df.columns]
if subset_cols:
    df = df.dropna(subset=subset_cols)
print(f"Row count after dropping missing appt_date / status: {len(df)}")

Shape: (1020, 14) | Duplicates: 20

Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  

Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [14]:
# (a) Treat any age below 1 or above 100 as invalid and set to missing
if "age" in df.columns:
    df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

# (b) Fill missing age with the median (then round & astype(int)) and missing distance_km with its median
if "age" in df.columns:
    median_age = df["age"].median()
    df["age"] = df["age"].fillna(median_age).round().astype(int)

if "distance_km" in df.columns:
    median_dist = df["distance_km"].median()
    df["distance_km"] = df["distance_km"].fillna(median_dist)

capped_counts = {}

# (c) Compute Q1, Q3, IQR and cap values with np.clip for distance_km and fee_inr
for col in ["distance_km", "fee_inr"]:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        num_capped = ((df[col] < lower_limit) | (df[col] > upper_limit)).sum()
        capped_counts[col] = num_capped

        # Cap values using np.clip
        df[col] = np.clip(df[col], lower_limit, upper_limit)

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int))
if "distance_km" in df.columns:
    df["distance_km"] = df["distance_km"].round(1)
if "fee_inr" in df.columns:
    df["fee_inr"] = df["fee_inr"].round().astype(int)

for col, count in capped_counts.items():
    print(f"Values capped in {col}: {count}")

Values capped in distance_km: 26
Values capped in fee_inr: 3


Task 1.4 — Features and a NumPy summary

In [15]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Mean of each column (axis=0)
column_means = np.mean(X, axis=0)

# (b) Number and share of rows with lead_days > 14 using a boolean mask
mask_lead = df["lead_days"] > 14
num_gt_14 = mask_lead.sum()
share_gt_14 = mask_lead.mean()

# (c) Z-scores of distance_km and count of rows with |z| > 2
distance_col = df["distance_km"].to_numpy()
z_scores = (distance_col - np.mean(distance_col)) / np.std(distance_col)
z_mask = np.abs(z_scores) > 2
count_z_gt_2 = z_mask.sum()

print(f"Column Means (age, distance_km, fee_inr, lead_days): {column_means}")
print(f"Lead days > 14 -> Count: {num_gt_14}, Share: {share_gt_14:.4f}")
print(f"Rows with distance z-score |z| > 2: {count_z_gt_2}")

Column Means (age, distance_km, fee_inr, lead_days): [ 48.17373737   9.50090909 635.71414141  11.81919192]
Lead days > 14 -> Count: 296, Share: 0.2990
Rows with distance z-score |z| > 2: 53


Task 1.5 — Enrich the clinics from a live API

In [16]:
temp_values = []

for idx, row in df.iterrows():
    clinic_name = row.get("clinic_name", f"Clinic_{idx}")
    lat = row.get("latitude")
    lon = row.get("longitude")

    if pd.isna(lat) or pd.isna(lon):
        print(f"Clinic '{clinic_name}': Missing latitude or longitude coordinates.")
        temp_values.append(None)
        continue

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }
    success = False
    temp_c = None

    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if "hourly" in data and "temperature_2m" in data["hourly"] and len(data["hourly"]["temperature_2m"]) > 0:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    success = True
                    break
                else:
                    reason = "Response JSON missing 'hourly' or 'temperature_2m' data."
            else:
                reason = f"HTTP status code {response.status_code}"
        except requests.exceptions.RequestException as e:
            reason = f"RequestException encountered: {e}"
        if attempt == 0:
            time.sleep(1)
    if not success:
        print(f"Clinic '{clinic_name}' failed to retrieve temperature. Reason: {reason}")
        temp_c = None
    temp_values.append(temp_c)

df["temp_c"] = temp_values
output_filename = "clinics_enriched.csv"
df.to_csv(output_filename, index=False)

print(f"\nEnriched dataset successfully saved to {output_filename}. Total records: {len(df)}")

Clinic 'Clinic_0': Missing latitude or longitude coordinates.
Clinic 'Clinic_1': Missing latitude or longitude coordinates.
Clinic 'Clinic_2': Missing latitude or longitude coordinates.
Clinic 'Clinic_4': Missing latitude or longitude coordinates.
Clinic 'Clinic_5': Missing latitude or longitude coordinates.
Clinic 'Clinic_6': Missing latitude or longitude coordinates.
Clinic 'Clinic_7': Missing latitude or longitude coordinates.
Clinic 'Clinic_8': Missing latitude or longitude coordinates.
Clinic 'Clinic_9': Missing latitude or longitude coordinates.
Clinic 'Clinic_10': Missing latitude or longitude coordinates.
Clinic 'Clinic_11': Missing latitude or longitude coordinates.
Clinic 'Clinic_12': Missing latitude or longitude coordinates.
Clinic 'Clinic_13': Missing latitude or longitude coordinates.
Clinic 'Clinic_14': Missing latitude or longitude coordinates.
Clinic 'Clinic_15': Missing latitude or longitude coordinates.
Clinic 'Clinic_16': Missing latitude or longitude coordinates.
C

Task 1.6 — Export the clean table and write the SQL load script

In [21]:
# STEP 1: Process & Save Appointments
if "appointment_id" in df.columns:
    df = df.sort_values("appointment_id").reset_index(drop=True)

for date_col in ["booked_on", "appt_date"]:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")

required_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"
]
df = df[required_columns]

print(f"Appointment ID is unique: {df['appointment_id'].is_unique}")
print(f"Total missing values across all 16 columns: {df.isnull().sum().sum()}")

df.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv successfully.")

# STEP 2: Create a Valid 7-Column Clinics File
clinic_info_map = {
    "C01": {"name": "Kothrud Family Clinic", "city": "Pune", "tier": "Tier 1", "lat": 18.5074, "lon": 73.8077},
    "C02": {"name": "Hadapsar Care Centre", "city": "Pune", "tier": "Tier 1", "lat": 18.5089, "lon": 73.9259},
    "C03": {"name": "Malviya Nagar Clinic", "city": "Jaipur", "tier": "Tier 2", "lat": 26.8549, "lon": 75.8243},
    "C04": {"name": "Vaishali Nagar Clinic", "city": "Jaipur", "tier": "Tier 2", "lat": 26.9124, "lon": 75.7447},
    "C05": {"name": "Vijay Nagar Clinic", "city": "Indore", "tier": "Tier 2", "lat": 22.7533, "lon": 75.8937},
    "C06": {"name": "Edappally Medical Centre", "city": "Kochi", "tier": "Tier 2", "lat": 10.0261, "lon": 76.3125},
    "C07": {"name": "Koramangala Health Hub", "city": "Bangalore", "tier": "Tier 1", "lat": 12.9352, "lon": 77.6245},
    "C08": {"name": "Andheri West Clinic", "city": "Mumbai", "tier": "Tier 1", "lat": 19.1136, "lon": 72.8258}
}

unique_clinics = df["clinic_id"].unique()
clinics_data = []
for cid in sorted(unique_clinics):
    info = clinic_info_map.get(cid, {"name": f"Clinic {cid}", "city": "Default City", "tier": "Tier 1", "lat": 18.5204, "lon": 73.8567})

    clinics_data.append({
        "clinic_id": cid,
        "clinic_name": info["name"],
        "city": info["city"],
        "tier": info["tier"],
        "latitude": info["lat"],
        "longitude": info["lon"],
        "temp_c": 28.0
    })

clinics_df = pd.DataFrame(clinics_data)
clinics_df.to_csv("clinics_enriched.csv", index=False)
print(f"Re-created 'clinics_enriched.csv' with exactly {len(clinics_df.columns)} columns.")

# STEP 3: Generate load_data.sql Safely
sql_lines = ["-- Insert statements for clinics"]
for _, row in clinics_df.iterrows():
    vals = ["NULL" if pd.isna(val) else (str(val) if isinstance(val, (int, float)) else f"'{val}'") for val in row]
    sql_lines.append(f"INSERT INTO clinics VALUES ({','.join(vals)});")

sql_lines.append("\n-- Insert statements for appointments")
for _, row in df.iterrows():
    vals = ["NULL" if pd.isna(val) else (str(val) if isinstance(val, (int, float)) else f"'{val}'") for val in row]
    sql_lines.append(f"INSERT INTO appointments VALUES ({','.join(vals)});")

with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines) + "\n")

print("Generated complete, error-free load_data.sql successfully!")

Appointment ID is unique: True
Total missing values across all 16 columns: 0
Saved appointments_clean.csv successfully.
Re-created 'clinics_enriched.csv' with exactly 7 columns.
Generated complete, error-free load_data.sql successfully!


In [ ]:
# Python Verification Script, Match counts with Pandas groupby

In [23]:

df_appts = pd.read_csv("appointments_clean.csv")
df_clinics = pd.read_csv("clinics_enriched.csv")

merged_df = pd.merge(df_appts, df_clinics, on="clinic_id", how="inner")

clinic_summary = merged_df.groupby(["clinic_name", "city"]).agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
).reset_index()

clinic_summary = clinic_summary[clinic_summary["appointments"] >= 95]

clinic_summary["no_show_pct"] = (100.0 * clinic_summary["no_shows"] / clinic_summary["appointments"]).round(1)
clinic_summary = clinic_summary.sort_values(by="no_show_pct", ascending=False)

print("Python Clinic Summary Validation (>= 95 appointments):")
print(clinic_summary.to_string(index=False))

Python Clinic Summary Validation (>= 95 appointments):
             clinic_name      city  appointments  no_shows  no_show_pct
    Hadapsar Care Centre      Pune           143        39         27.3
   Vaishali Nagar Clinic    Jaipur           142        38         26.8
   Kothrud Family Clinic      Pune           160        36         22.5
  Koramangala Health Hub Bangalore           101        21         20.8
Edappally Medical Centre     Kochi           142        23         16.2
      Vijay Nagar Clinic    Indore           130        18         13.8
